In [1]:
import os
import polars as pl
from dataclasses import dataclass

PROJECT_PATH = "/data1/datasets_1/human_cistrome/chip-atlas/peak_calls/tfbinding_scripts/tf-binding"

@dataclass
class SampleConfig:
    label: str
    sample: str
    ground_truth_file: str

# /data1/datasets_1/human_cistrome/chip-atlas/peak_calls/tfbinding_scripts/tf-binding/data/processed_results/AR-PC3_PC-3_processed.parquet
# create a list of SampleConfig objects
sample_configs = [
    SampleConfig(label="AR-PC3", sample="PC-3", ground_truth_file=f"{PROJECT_PATH}/data/transcription_factors/AR/merged/PC-3_AR_merged.bed"),

]


In [3]:
import tempfile
from src.utils.generate_training_peaks import run_bedtools_command
def intersect_bed_files(main_df: pl.DataFrame, intersect_df: pl.DataFrame, region_type: str = None) -> pl.DataFrame:
    """
    Intersect two BED files using bedtools and return the original DataFrame with overlap flags.
    
    Args:
        main_df: Primary Polars DataFrame with BED data
        intersect_df: Secondary Polars DataFrame to intersect with
        region_type: Optional region type label to add to results
        
    Returns:
        Original DataFrame with additional column indicating overlaps
    """
    with tempfile.NamedTemporaryFile(delete=False, mode='w') as main_file, \
         tempfile.NamedTemporaryFile(delete=False, mode='w') as intersect_file, \
         tempfile.NamedTemporaryFile(delete=False, mode='w') as result_file:
        
        main_path = main_file.name
        intersect_path = intersect_file.name
        result_path = result_file.name

        # Write DataFrames to temporary files
        main_df.write_csv(main_path, separator="\t", include_header=False)
        intersect_df.write_csv(intersect_path, separator="\t", include_header=False)

        # Run bedtools intersect with -c flag to count overlaps
        command = f"bedtools intersect -a {main_path} -b {intersect_path} -c > {result_path}"
        run_bedtools_command(command)

        # Read results back into Polars DataFrame
        result_df = pl.read_csv(
            result_path,
            separator="\t",
            has_header=False,
            new_columns=[*main_df.columns, "overlap_count"]
        )

    # Clean up temporary files
    os.remove(main_path)
    os.remove(intersect_path) 
    os.remove(result_path)

    # Add boolean overlap column
    result_df = result_df.with_columns(
        pl.col("overlap_count").gt(0).alias("overlaps_ground_truth")
    ).drop("overlap_count")

    return result_df


In [4]:
# Add dfs to list
HIGH_COUNT_QUANTILE = 0.75
MAX_COUNT_THRESHOLD = 30
MID_COUNT_THRESHOLD = 10


def threshold_peaks(df):
    max_count = df.select(pl.col("count").max()).item()
    
    if max_count <= 2:
        return df
    elif max_count > MAX_COUNT_THRESHOLD:
        threshold = df.select(pl.col("count").quantile(HIGH_COUNT_QUANTILE)).item()
        df = df.filter(pl.col("count") > threshold)
    elif max_count > MID_COUNT_THRESHOLD:
        threshold = df.select(pl.col("count").median()).item()
        df = df.filter(pl.col("count") > threshold)
    return df

dfs = []
for sample_config in sample_configs:
    parquet_path = PROJECT_PATH + "/data/processed_results/" + sample_config.label + "_" + sample_config.sample + "_processed.parquet"
    # print(parquet_path)
    df = pl.read_parquet(parquet_path, columns=["chr_name", "start", "end", "cell_line", "targets", "predicted", "weights", "probabilities"])
    df = df.rename({"chr_name": "chr"})
    print(df)
    chip_data = pl.read_csv(sample_config.ground_truth_file, separator="\t", has_header=False, new_columns=["chr", "start", "end"])
    intersected_df = intersect_bed_files(df[["chr", "start", "end"]], chip_data)
    ground_truth_df = df.join(intersected_df, on=["chr", "start", "end"], how="left")
    ground_truth_df = ground_truth_df.with_columns(pl.when(pl.col("overlaps_ground_truth")).then(1).otherwise(0).alias("targets"))
    ground_truth_df = ground_truth_df.drop("overlaps_ground_truth")
    dfs.append(ground_truth_df)

dfs[0]


shape: (67_537, 8)
┌───────┬───────────┬───────────┬───────────┬─────────┬───────────┬─────────┬───────────────┐
│ chr   ┆ start     ┆ end       ┆ cell_line ┆ targets ┆ predicted ┆ weights ┆ probabilities │
│ ---   ┆ ---       ┆ ---       ┆ ---       ┆ ---     ┆ ---       ┆ ---     ┆ ---           │
│ str   ┆ i64       ┆ i64       ┆ str       ┆ f64     ┆ f64       ┆ f64     ┆ f64           │
╞═══════╪═══════════╪═══════════╪═══════════╪═════════╪═══════════╪═════════╪═══════════════╡
│ chr10 ┆ 73439293  ┆ 73440261  ┆ PC-3      ┆ -1.0    ┆ 0.0       ┆ -1.0    ┆ 0.015723      │
│ chr4  ┆ 116400326 ┆ 116400879 ┆ PC-3      ┆ -1.0    ┆ 0.0       ┆ -1.0    ┆ 0.012411      │
│ chr1  ┆ 223504798 ┆ 223505120 ┆ PC-3      ┆ -1.0    ┆ 0.0       ┆ -1.0    ┆ 0.002116      │
│ chr6  ┆ 35919199  ┆ 35921364  ┆ PC-3      ┆ -1.0    ┆ 0.0       ┆ -1.0    ┆ 0.019968      │
│ chr9  ┆ 96382686  ┆ 96383841  ┆ PC-3      ┆ -1.0    ┆ 0.0       ┆ -1.0    ┆ 0.011403      │
│ …     ┆ …         ┆ …         ┆ …      

chr,start,end,cell_line,targets,predicted,weights,probabilities
str,i64,i64,str,i32,f64,f64,f64
"""chr10""",73439293,73440261,"""PC-3""",0,0.0,-1.0,0.015723
"""chr4""",116400326,116400879,"""PC-3""",0,0.0,-1.0,0.012411
"""chr1""",223504798,223505120,"""PC-3""",0,0.0,-1.0,0.002116
"""chr6""",35919199,35921364,"""PC-3""",0,0.0,-1.0,0.019968
"""chr9""",96382686,96383841,"""PC-3""",0,0.0,-1.0,0.011403
…,…,…,…,…,…,…,…
"""chr6""",157221125,157221781,"""PC-3""",0,0.0,-1.0,0.043681
"""chr1""",31135850,31136365,"""PC-3""",0,0.0,-1.0,0.002725
"""chr14""",101980171,101980949,"""PC-3""",0,0.0,-1.0,0.000601


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve, auc, f1_score, confusion_matrix
import matplotlib.pyplot as plt

def ensure_numpy(arr):
    """Convert input to numpy array if it isn't already."""
    return np.array(arr) if not isinstance(arr, np.ndarray) else arr

def find_best_f1_threshold(y_true, y_score):
    """Find the optimal threshold that gives the best F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-7)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
    return best_threshold, f1_scores[best_idx]

def get_display_name(config):
    """Get display name based on sample and label if available."""
    return f"{config.label} ({config.sample})" if hasattr(config, 'label') and config.label else config.sample

def print_confusion_metrics(dfs, sample_configs, threshold=None):
    """
    Print confusion matrix metrics for multiple datasets.
    
    Args:
        dfs: List of dataframes containing targets and probabilities
        sample_configs: List of sample configurations for labeling
        threshold: Optional fixed threshold to use. If None, finds best F1 threshold
    """
    print("\nConfusion Matrix Metrics:")
    print("-" * 140)
    headers = ["Dataset", "Threshold", "True Negative", "False Positive", "False Negative", "True Positive", "Total", "Accuracy"]
    print(f"{headers[0]:<25} {headers[1]:<10} {headers[2]:<15} {headers[3]:<15} {headers[4]:<15} {headers[5]:<15} {headers[6]:<10} {headers[7]:<10}")
    print("-" * 140)
    
    for df, config in zip(dfs, sample_configs):
        y_true = ensure_numpy(df["targets"])
        y_score = ensure_numpy(df["probabilities"])
        
        # Determine threshold
        if threshold is None:
            best_threshold, _ = find_best_f1_threshold(y_true, y_score)
            used_threshold = best_threshold
        else:
            used_threshold = threshold
        
        # Get predictions using threshold
        y_pred = (y_score >= used_threshold).astype(int)
        
        # Calculate confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        total = tn + fp + fn + tp
        accuracy = (tp + tn) / total
        
        display_name = get_display_name(config)
        print(f"{display_name:<25} {used_threshold:>9.3f} {tn:>15} {fp:>15} {fn:>15} {tp:>15} {total:>10} {accuracy:>9.3f}")
    
    print("-" * 140)

# Use with fixed threshold can also do None for best F1 threshold
threshold = .95 # this is the threshold we have used for almost all AR models z(see TF cell lines file in this directory)
print_confusion_metrics(dfs, sample_configs, threshold=threshold)


Confusion Matrix Metrics:
--------------------------------------------------------------------------------------------------------------------------------------------
Dataset                   Threshold  True Negative   False Positive  False Negative  True Positive   Total      Accuracy  
--------------------------------------------------------------------------------------------------------------------------------------------
AR-PC3 (PC-3)                 0.950           66302             952             214              69      67537     0.983
--------------------------------------------------------------------------------------------------------------------------------------------
